# Lolipop

In [39]:
import plotly.graph_objects as go
import pandas as pd
import plotly.express as px
import json
import requests
from functools import reduce
path = "../datos"
#path = "/content"

In [40]:
df = pd.read_excel(path + "/1.3.1.A_sh_es.xls", sheet_name=0, header = None)
df = df.iloc[7:-42] # Eliminar todos los renglones de texto, solo dejar el nacional
df.columns = df.iloc[0] # El encabezado es el primer renglón
#df = df[1:] # Eliminar el primer renglon (ya es el encazabezado)
df.columns = ["cvegeo","Entidad", "Sin seguridad social", "2020", "2018", "2016"]
df_SS = df.iloc[:, 0:3]
df_SS.iloc[0, 2] = 100 - df_SS.iloc[0, 2]
# Usar el complemento, el numero actual es la gente que tiene SS
# usar el complemento on 100, para saber la gente que NO tiene SS (Carencia Social)
df_SS

,cvegeo,Entidad,Sin seguridad social
7,00,Estados Unidos Mexicanos,50.183773


In [41]:
df = pd.read_excel("/content/2N.2.1.A_sh_es.xls", sheet_name=0, header = None)
df = df.iloc[9:-53] # Eliminar todos los renglones de texto, solo dejar el nacional
df.columns = df.iloc[0] # El encabezado es el primer renglón
#df = df[1:] # Eliminar el primer renglon (ya es el encazabezado)
df_Alim = df.iloc[:, 0:3]
df_Alim.columns = ["cvegeo","Entidad", "Con Inseguridad Alimentaria"]
extraer = df_Alim['Con Inseguridad Alimentaria'].str.extract(r'(?P<Numero>[0-9.]+)\s*(?P<Sufijo>.*)')
df_Alim['Con Inseguridad Alimentaria'] = pd.to_numeric(extraer['Numero'], errors="coerce")
df_Alim


/tmp/ipykernel_14943/4095242160.py:8: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,cvegeo,Entidad,Con Inseguridad Alimentaria
9,00,Estados Unidos Mexicanos,16.234029


In [42]:
df = pd.read_excel("/content/6N.1.1.C_sh_es.xls", sheet_name=0, header = None)
df = df.iloc[7:-35] # Eliminar todos los renglones de texto, solo dejar el nacional
df.columns = df.iloc[0] # El encabezado es el primer renglón
#df = df[1:] # Eliminar el primer renglon (ya es el encazabezado)
df_Agua = df.iloc[:, 0:3]
df_Agua.columns = ["cvegeo","Entidad", "Sin agua potable y drenaje"]
# Usamos el complemento, esta es la porporción de la poblacion
# que tiene AGua y Sanemaiento; considerar el complemento para
# englobar alguna de las dos carencias
df_Agua.iloc[0, 2] = 100 - df_Agua.iloc[0, 2]
df_Agua

,cvegeo,Entidad,Sin agua potable y drenaje
7,00,Estados Unidos Mexicanos,44.134784


In [43]:
df = pd.read_excel("/content/4.1.2_sh_es.xls", sheet_name=0, header = None)
df = df.iloc[7:-39] # Eliminar todos los renglones de texto
df.columns = df.iloc[0] # El encabezado es el primer renglón
df = df[1:] # Eliminar el primer renglon (ya es el encazabezado)
df = df.iloc[1:, [0,1,32]]
df_Prepa = df.iloc[:, 0:3]
df_Prepa.columns = ["cvegeo","Entidad", "No terminó la prepa"]
# Usamos el complemento, esta es la porporción de la poblacion
# que finalizó la prepa en la edad que debía; considerar el complemento para
# englobar la pob que no terminó a la edad que debia
df_Prepa['No terminó la prepa'] = pd.to_numeric(df_Prepa["No terminó la prepa"], errors="coerce")
df_Prepa.iloc[0, 2] = 100 - df_Prepa.iloc[0, 2]
df_Prepa

,cvegeo,Entidad,No terminó la prepa
9,00,Estados Unidos Mexicanos,35.851873


In [44]:
dfs = [df_SS, df_Agua, df_Prepa, df_Alim]

# Unimos todos usando 'Año' y 'Nivel' como anclas
df = reduce(lambda left, right: pd.merge(left, right, on=['cvegeo', 'Entidad']), dfs)
df = df.iloc[:, 2:6]
# Ordenar de mayor a menor para mejor visualización
#df = df.sort_values(by='Valor', ascending=True)

df_long = df.melt(var_name='Indicador', value_name='Valor')

# Ordenar para que la gráfica sea legible
df_long = df_long.sort_values(by='Valor', ascending=True)

# 3. Crear la gráfica Lollipop
fig = go.Figure()

# Tallos
for i, row in df_long.iterrows():
    fig.add_shape(
        type='line',
        x0=0, y0=row['Indicador'],
        x1=row['Valor'], y1=row['Indicador'],
        line=dict(color='#b90095', width=1.3)
    )

# Puntos
fig.add_trace(go.Scatter(
    x=df_long['Valor'],
    y=df_long['Indicador'],
    mode='markers',
    marker=dict(color='#b90095', size=17),
    hovertemplate='<b>%{y}</b><br>Porcentaje: %{x:.2f}%<extra></extra>'
))

# Diseño
fig.update_layout(
    title='Carencias sociales en México (2022): porcentaje de población afectada <br><sup>INEGI-CONEVAL, ENIGH 2022</sup>',
    xaxis_title='<b>Porcentaje (%)<b>',
    yaxis_title='',
    yaxis=dict(
        tickfont=dict(family='Arial', size=15, color='black'),
        tickformat="", # Asegura que se respete el texto
    ),
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='#fff0f5'),
    height=500,
    width=800
)

#fig.update_yaxes(ticktext=["<b>" + str(label) + "</b>" for label in df_long['Indicador'].unique()],
#                 tickvals=df_long['Indicador'].unique())

fig.show()